In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import pandas as pd
import datetime as dt

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.merger import Merger


In [3]:
org_path = "./all_orgs.pkl"
client_path = "./all_clicnts.pkl"
elv_plans_path = "./all_elv_plans.pkl"
cu_plans_path = "./all_cu_plans.pkl"


org_df = pd.read_pickle(org_path)
print(len(org_df))
client_df = pd.read_pickle(client_path)
print(len(client_df))
elv_plans_df = pd.read_pickle(elv_plans_path)
print(len(elv_plans_df))
cu_plans_df = pd.read_pickle(cu_plans_path)
print(len(cu_plans_df))

1598
2761
5883
6175


In [4]:
[c for c in org_df.columns.to_list() + elv_plans_df.columns.to_list() if "auto" in c]

['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment',
 'plan_account_funding_config.is_auto_enrollment.is_auto_enrollment_state']

In [5]:
cu_plans_df[cu_plans_df['rollover_eligibility'].notna()]

,cu_plan_id,custom_id,custom_item_id,name,text_content,cu_plan_description,date_created,date_updated,date_closed,date_done,...,fund_ado,inv_ado,claim_ado,run_out_end_formula,grace_period_end_formula,run_out_benefit_termed_end_formula,client_edu,fund_edu,inv_edu,claim_edu
136,868e31ne6,None,1002,RMRCTC FSA 2025,,,2025-05-23 22:37:17.143,2025-05-23 22:37:19.189,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
676,868c5fq6h,None,1002,RMRFED FSA 2025,,,2025-01-22 14:11:31.316,2025-04-29 15:23:05.570,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
974,868anng84,None,1002,RMROFA FSA 2025,,,2024-11-06 14:40:28.465,2025-07-10 11:35:51.016,None,2024-11-14 01:38:35.285,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2212,86879kru4,None,1002,RMRGFC FSA 2024,*LPF Plan added effective 1/1/23 with HSA plan...,*LPF Plan added effective 1/1/23 with HSA plan...,2024-02-08 01:02:22.645,2025-04-29 17:36:45.109,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2371,86879kr83,None,1002,RMRCTC FSA 2024,,,2024-02-08 00:54:09.509,2025-04-29 15:20:07.419,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2409,86879bdjc,None,1002,RMRSNOW FSA 2030,RMRSNOW FSA 2024,RMRSNOW FSA 2024,2024-02-07 09:58:23.618,2025-08-19 11:22:09.345,None,2024-04-11 10:35:24.691,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5635,8687acgnv,None,1002,RMRCOR HRA 2024,City of Rifle\nSPD\n213-D\nSpecial Instruction...,City of Rifle\nSPD\n213-D\nSpecial Instruction...,2024-02-11 00:47:06.221,2025-06-06 11:05:59.987,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
merger = Merger(
    organization_df=org_df,
    elv_plan_df=elv_plans_df,
    client_df=client_df,
    cu_plan_df=cu_plans_df
)

merged_df = merger.merge_all()


In [7]:
adjustments_df = merged_df.copy()

adjustments_df['plan_coverage_config.grace_period_type.grace_period_days_amount'] = adjustments_df.apply(
    lambda row:
    75 if
        row['plan_coverage_config.grace_period_type.grace_period_type'] == "TWO_AND_HALF_MONTH"
    else
        row['plan_coverage_config.grace_period_type.grace_period_days_amount'],
    axis=1
)

adjustments_df['(Elevate) Carryover only through next Plan Year'] = adjustments_df.apply(
    lambda row:
        "" if
            pd.isna(row['plan_account_funding_config.is_rollover.is_rollover']) or
            pd.isna(row['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment'])
        else True if
            row['plan_account_funding_config.is_rollover.is_rollover'] == True and
            row['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment'] == False
        else False,
    axis=1
)

adjustments_df['plan_primary_config.max_election_amount_type.max_election_amount'] = adjustments_df.apply(
    lambda row:
        "UNLIMITED"
        if
            row['plan_primary_config.max_election_amount_type.max_election_amount_type'] == "UNLIMITED"
        else
            row['plan_primary_config.max_election_amount_type.max_election_amount'],
    axis=1
)

adjustments_df['plan_account_funding_config.max_rollover_amount.max_rollover_amount'] = adjustments_df.apply(
    lambda row:
        "UNLIMITED"
        if 
            row['plan_account_funding_config.max_rollover_amount.max_rollover_amount_type'] == "UNLIMITED"
        else
            row['plan_account_funding_config.max_rollover_amount.max_rollover_amount'],
    axis=1
)


In [8]:
cols_to_drop = [
    "inv_pkg",
    "claim_trn",
    "fund_trn",
    "fund_ado",
    "inv_ado",
    "claim_ado",
    "fund_edu",
    "claim_edu",
    "claim_hra",
    "client_hra_y",
    "fund_hra",
    "inv_hra",
    "client_pkg_y",
    "client_trn_y",
    "claim_lsa",
    "client_lsa_y",
    "fund_lsa",
    "inv_lsa",
    "inv_edu",
    "client_ado_y",
    "fund_pkg",
    "claim_pkg",
    'client_fsa_x',
    'client_hsa_x',
    'client_dca_x',
    'client_hra_x',
    'client_lsa_x',
    'client_pkg_x',
    'client_cobra',
    'client_data',
    'client_ado_x',
    'client_edu_x',
    'client_inv',
    'client_fund',
    'client_elv',
    'claim_dca',
    'claim_fsa',
    'claim_hsa',
    'client_dca_y',
    'client_edu_y',
    'client_fsa_y',
    'client_hsa_y',
    'client_id',
    'client_id_list',
    'client_name',
    'client_trn_x',
    "account_type.account_type_state",
    "points",
    'team_id',
    'run_out_benefit_termed_end_formula',
    'run_out_benefit_termed_end_formula',
    'run_out_benefit_termed_end_formula',
    'run_out_benefit_termed_end_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'plan_primary_config.fund_id_formula.fund_id_formula',
    'plan_primary_config.fund_id_formula.fund_id_formula_state',
    'grace_period_end_formula',
    'loyalty_formula',
    'run_out_end_formula',
    'run_out_end_formula',
    'grace_period_end_formula',
    'loyalty_formula',
    'loyalty_formula',
    'run_out_end_formula',
    'grace_period_end_formula',
    'loyalty_formula',
    'run_out_end_formula',
    'run_out_benefit_termed_end_formula',
    'grace_period_end_formula',
    'run_out_end_formula',
    'run_out_benefit_termed_end_formula',
    'run_out_end_formula',
    'grace_period_end_formula',
    'run_out_benefit_termed_end_formula',
    'run_out_end_formula',
    'grace_period_end_formula',
    'run_out_benefit_termed_end_formula',
    'creator.color_x',
    'creator.color_y',
    'creator.email_x',
    'creator.email_y',
    'creator.id_x',
    'creator.id_y',
    'creator.profilePicture_x',
    'creator.profilePicture_y',
    'creator.username_x',
    'creator.username_y',
    'terminated_plan_types',
    'termination_date',
    'termination_details',
    'termination_reason',
    'text_content',
    'tier',
    'time_estimate',
    'time_spent',
    'update_account_managers',
    'watchers',
    'wave',
    'website',
    'prior_plan.account_type',
    'prior_plan.account_type_state',
    'prior_plan.deletable',
    'prior_plan.end_of_coverage_type',
    'prior_plan.id',
    'prior_plan.name',
    'prior_plan.name_state',
    'prior_plan.organization_id',
    'prior_plan.plan_code',
    'prior_plan.plan_status',
    'prior_plan.plan_year.id',
    'prior_plan.plan_year.name',
    'prior_plan.termination_properties',
    'prior_plan_id',
    'previous_administrator',
    'previous_names',
    'plan_year.id',
    'plan_year.name',
    'plan_year.organization_id',
    'plan_year.organization_path',
    'plan_year.prior_plan_year.id',
    'plan_year.prior_plan_year.name',
    'plan_year.prior_plan_year.valid_from',
    'plan_year.prior_plan_year.valid_to',
    'plan_year.prior_plan_year_id',
    'structure',
    'temp_account_manager',
    'temp_am_cobra',
    'short_name',
    'social_facebook',
    'social_linkedin',
    'social_twitter',
    "start_date",
    "type",
    'plan_omnibus_account_id',
    'plan_organization_path',
    'priority',
    'priority.color',
    'priority.id',
    'priority.orderindex',
    'priority.priority',
    'divisions',
    'ducks',
    'due_date',
    "deletable",
    'cu_plan_assignees',
    'cu_plan_assignees',
    'status.color_x',
    'status.color_y',
    'status.id_x',
    'status.id_y',
    'status.orderindex_x',
    'status.orderindex_y',
    'status.type_x',
    'status.type_y',
    'inv_dca',
    'inv_fsa',
    'inv_hsa',
    'fund_dca',
    'fund_fsa',
    'fund_hsa',
    'date_updated',
    'date_closed',
    'date_created',
    'date_done',
    "date_benefit_termed",
    "plan_year_id",
    'other_description',
    'parent',
    'permission_level',
    "notes",
    'top_level_parent',
    'service_configs.ADOPTION',
    'service_configs.DENTAL',
    'service_configs.DEPENDENT_CARE',
    'service_configs.LIFE_STYLE',
    'service_configs.MEDICAL',
    'service_configs.PARKING',
    'service_configs.PHARMACY',
    'service_configs.PLAN_SPECIFIC',
    'service_configs.PREMIUM',
    'service_configs.TRANSIT',
    'service_configs.TRAVEL',
    'service_configs.VISION',
    'service_configs.WELLNESS',
    'cu_client_archived',
    'cu_client_assignees',
    'cu_client_assignees',
    'cu_client_description',
    'cu_client_elv_id',
    'cu_client_tags',
    'cu_client_url',
    'cu_plan_archived',
    'cu_plan_description',
    'cu_plan_elv_id',
    'cu_plan_tags',
    'cu_plan_url',
    'data_end',
    'data_start',
    'checklists',
    'client_brokerage',
    'client_contacts',
    'client_project',
    "cobra_manager",
    'created_at',
    'custom_id',
    'custom_item_id',
    'divisional_invoicing',
    'elevate_auto_renew',
]

rename_map = {
    "rmrcode": "RMRCODE",
    
    "organization_id": "Organization ID",
    "client_id": "Client ID",

    "cu_client_status": "Client Status",
    "organization_status_type": "Organization Status",

    "cu_client_account_manager": "(ClickUp) Client Account Manager",
    "cu_plan_account_manager": "(ClickUp) Plan Account Manager",

    "elv_plan_id": "(Elevate) Plan ID",
    "cu_plan_id": "(ClickUp) Plan ID",

    "cu_plan_id": "(ClickUp) Plan ID",
    "elv_plan_id": "(Elevate) Plan ID",

    "account_type.account_type": "(Elevate) Plan Type",
    "cu_account_type": "(ClickUp) Plan Type",

    "elv_plan_status": "(Elevate) Plan Status",
    "cu_plan_status": "(ClickUp) Plan Status",

    "plan_year.valid_from": "(Elevate) Date Plan Start",
    "date_plan_start": "(ClickUp) Date Plan Start",

    "plan_year.valid_to": "(Elevate) Date Plan End",
    "date_plan_end": "(ClickUp) Date Plan End",
    
    "plan_code": "(Elevate) Plan Code",
    "elv_plan_code": "(ClickUp) Plan Code",

    "plan_primary_config.max_election_amount_type.max_election_amount": "(Elevate) Max. Annual Election",
    "annual_election_max": "(ClickUp) Max. Annual Election",

    "plan_primary_config.min_election_amount_type.min_election_amount": "(Elevate) Min. Annual Election",
    "annual_election_min": "(ClickUp) Min. Annual Election",
    
    "annual_election_auto_adjust": "(ClickUp) Annual Election Auto Adjust",

    "auto_post_contributions": "(ClickUp) Auto Post Contributions",

    "plan_account_funding_config.is_rollover.is_rollover": "(Elevate) Has Rollover",
    "rollover": "(ClickUp) Has Rollover",

    "plan_account_funding_config.max_rollover_amount.max_rollover_amount": "(Elevate) Max. Rollover",
    "rollover_max": "(ClickUp) Max. Rollover",
    
    "plan_account_funding_config.min_rollover_amount.min_rollover_amount": "(Elevate) Min. Rollover",
    "rollover_min": "(ClickUp) Min. Rollover",

    'plan_account_funding_config.rollover_claims.rollover_claims_days_amount': "(Elevate) Rollover Claims Days",
    
    "rollover_eligibility": "(ClickUp) Rollover Eligibility",

    "plan_account_funding_config.is_auto_enrollment.is_auto_enrollment": "(Elevate) Auto Enrollment",
    "(Elevate) Carryover only through next Plan Year": "(Elevate) Carryover only through next Plan Year",

    "plan_coverage_config.grace_period_type.grace_period_days_amount": "(Elevate) Grace Period Days",
    "grace_period": "(ClickUp) Grace Period",

    'plan_coverage_config.run_out_type.run_out_days_amount': "(Elevate) Run Out Days",
    "run_out": "(ClickUp) Has Run Out",
    "run_out_termed_ee": "(ClickUp) Run Out Termed EE",
    "run_out_termed_benefit": "(ClickUp) Run Out Termed Benefit",

    'plan_coverage_config.end_of_coverage_type.end_of_coverage_days_amount': "(Elevate) End of Coverage Days",

    "lpf": "(ClickUp) Limited Purpose Offered", 

    "card": "(ClickUp) Card Offered",
    "plan_primary_config.is_carded.is_carded": "(Elevate) Card Offered",

    "cu_plan_account_manager": "(ClickUp) Plan Account Manager",

    "cu_plan_date_admin_start": "(ClickUp) Plan Date Admin Start",
    
    "cu_client_date_admin_start": "(Clickp) Client Date Admin Start",
    # "cu_account_type": "(ClickUp) Account Type",

    "bank": "(ClickUp) HSA Bank",

    "year_oe_approved": "(ClickUp) Year OE Approved",
}

print(f"incorrect rename map: {[c for c in rename_map.keys() if c not in adjustments_df.columns]}")

columns_to_map = sorted([c for c in adjustments_df.columns if c not in list(rename_map.keys()) + cols_to_drop])

print(f"{len(columns_to_map)} more to go:")
display(columns_to_map)



incorrect rename map: []
202 more to go:


['activation_started',
 'address',
 'address_billing',
 'address_city',
 'address_mail',
 'address_state',
 'address_street',
 'address_street_line_',
 'address_zip',
 'broker_involvement',
 'business_entity_type',
 'data_transmission_details',
 'date_rate_expiration',
 'date_renewal_year_begins',
 'date_renewal_year_ends',
 'dependencies',
 'eligible_employee_count',
 'elv_org_parent_id',
 'elv_plan_name',
 'elv_plan_parent_id',
 'email_domains',
 'employer_contributions',
 'forfeiture_date',
 'grace_termed_ee',
 'grace_termed_ee_days',
 'health_plan_codes',
 'hq_city',
 'hq_state',
 'hsa_bank',
 'initial_deposit',
 'is_forfeited',
 'is_limited_features_during_termination_in_progress',
 'is_locked',
 'is_plan',
 'linked_tasks',
 'list.access_x',
 'list.access_y',
 'list.id_x',
 'list.id_y',
 'list.name',
 'locations',
 'name',
 'name.name_state',
 'name_operational',
 'name_umbrella',
 'notional_funding_account_id',
 'notional_payroll_account_id',
 'opportunity_client',
 'organization

In [9]:
currency_columns = [
    "(Elevate) Max. Annual Election",
    "(ClickUp) Max. Annual Election",
    "(Elevate) Min. Annual Election",
    "(ClickUp) Min. Annual Election",
    "(Elevate) Max. Rollover",
    "(ClickUp) Max. Rollover",
    "(Elevate) Min. Rollover",
    "(ClickUp) Min. Rollover",
]

status_columns = [
    "Client Status",
    "Organization Status",
    "(Elevate) Plan Status",
    "(ClickUp) Plan Status"
]

In [10]:
renamed_df = adjustments_df.rename(columns=rename_map)
reordered_df = renamed_df[rename_map.values()]
sorted_df = reordered_df.sort_values(
    by=['RMRCODE', '(Elevate) Plan Type', '(Elevate) Date Plan Start'], 
    ascending=[True, True, False]
)

formatted_df = sorted_df.copy()

for col in [c for c in formatted_df.columns if "date" in c.lower()]:
    print(col)
    formatted_df[col] = formatted_df[col].apply(lambda x: dt.datetime.strftime(x, "%m/%d/%Y") if isinstance(x, dt.datetime) and pd.notna(x) else "")

print(f"missing currency columns: {[c for c in currency_columns if c not in formatted_df.columns]}")

def format_currency(value):
    if value == "" or pd.isna(value):
        return ""
    
    try:
        clean_val = str(value).replace(",", "").replace("$", "").strip()
        return f"{float(clean_val):.2f}"
    except (ValueError, AttributeError):
        return str(value)

for col in currency_columns:
    formatted_df[col] = formatted_df[col].apply(lambda x: format_currency(x))


for col in status_columns:
    formatted_df[col] = formatted_df[col].apply(lambda x: str(x).lower() if pd.notna(x) else "")

formatted_df = formatted_df.fillna("")
formatted_df = formatted_df.replace("None", "")
# missing_columns = [c for c in rename_map.values() if c not in rename_map]
# display(sorted_df[[ "RMRCODE", "Organization ID", "Client ID", "Client Status", "Organization Status", "(Elevate) Plan Type", "(Elevate) Date Plan Start"]])

# display(sorted_df)


(Elevate) Date Plan Start
(ClickUp) Date Plan Start
(Elevate) Date Plan End
(ClickUp) Date Plan End
(ClickUp) Plan Date Admin Start
(Clickp) Client Date Admin Start
missing currency columns: []


In [11]:
more_columns_to_drop = [
    c for c in formatted_df if "clickup" in c.lower() 
] + [
    c for c in formatted_df if "client" in c.lower()
]

narrow_df = formatted_df.drop(columns=more_columns_to_drop)
narrow_df.to_excel("narrow_df.xlsx", index=False, na_rep='')

In [12]:
cols_to_compare = [c for c in formatted_df.columns if "(" in c]
base_names = {c.split(")")[1].strip() for c in cols_to_compare}

new_columns = {}

for base_name in base_names:
    col_search = [c for c in formatted_df.columns if base_name in c]
    
    if len(col_search) != 2:
        continue
    
    col1, col2 = col_search
    max_idx = max(formatted_df.columns.get_loc(col1), 
                   formatted_df.columns.get_loc(col2))
    
    new_column = f"{base_name} Matches"
    match_series = formatted_df.apply(
        lambda row:
            "" if any([row[col1] == "", row[col2] == ""])
            else row[col1] == row[col2],
        axis=1
    )
    
    new_columns[max_idx + 1] = (new_column, match_series)

matching_df = formatted_df.copy()
for position in sorted(new_columns.keys(), reverse=True):
    col_name, col_data = new_columns[position]
    matching_df.insert(position, col_name, col_data)
    # print(column_indices)


    # print(f"{col}:")
    # for c in col_search:
    #     print(f"\t{c}")
    
    # print(f"\t{col} Matches")

    # print("\n")

In [13]:
matching_df.to_excel("matched_df.xlsx", index=False, na_rep='')

In [ ]:

# platform_columns = [c for c in sorted_df.columns.to_list() if "(" in c]
# platform_columns = set([c.split(")")[1].strip() for c in platform_columns])

# matching_pairs = []
# for col in platform_columns:
#     col_search = sorted([c for c in sorted_df.columns if col in c])

#     if len(col_search) != 2:
#         print(col)
#         continue


#     matching_pairs.append((col_search[1], col_search[0]))

    

# matching_pairs
